# Multi-Horizon GBM Forecasting for Attraction Wait Times

Brief methodology:
- Direct multi-horizon: train one model per horizon h = 1..7.
- Leakage-safe features: all lags and rolling stats are computed with shift(1).
- Known-future exogenous: use t+h values for season, dow, weather, and events.
- Time-respecting validation: expanding-window walk-forward splits plus a final holdout.
- Uncertainty: quantile models if supported, otherwise conformal intervals on residuals.


In [ ]:
import json
import os
import random
import warnings
from pathlib import Path
from importlib import metadata

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ----------------------------
# 1) Configuration
# ----------------------------
DATE_COL = "date"
ENTITY_COL = "ENTITY_DESCRIPTION_SHORT"
TARGET_COL = "wait_time_avg"
HORIZONS = [1, 2, 3, 4, 5, 6, 7]
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# Prefer LightGBM, fallback to XGBoost if not available
USE_LGBM = False
try:
    import lightgbm as lgb
    USE_LGBM = True
except Exception:
    lgb = None

if not USE_LGBM:
    try:
        import xgboost as xgb
    except Exception as e:
        raise ImportError("Install lightgbm or xgboost to run this notebook.") from e


def _version(pkg_name):
    try:
        return metadata.version(pkg_name)
    except Exception:
        return "not installed"

print("pandas:", _version("pandas"))
print("numpy:", _version("numpy"))
print("lightgbm:", _version("lightgbm"))
print("xgboost:", _version("xgboost"))
print("sklearn:", _version("scikit-learn"))
print("Using LightGBM" if USE_LGBM else "Using XGBoost")


pandas: 3.0.0
numpy: 2.4.2
lightgbm: 4.6.0
xgboost: 3.2.0
sklearn: 1.8.0
Using LightGBM


## 2) Load data and sanity checks

Assumes a DataFrame named `model_df` is available. If not, load from a placeholder path.


In [2]:
# ----------------------------
# 2) Load data / sanity checks
# ----------------------------
DATA_PATH_PARQUET = Path("data/model_df.parquet")  # placeholder
DATA_PATH_CSV = Path("data/model_df.csv")          # placeholder

if "model_df" not in globals():
    if DATA_PATH_PARQUET.exists():
        model_df = pd.read_parquet(DATA_PATH_PARQUET)
    elif DATA_PATH_CSV.exists():
        model_df = pd.read_csv(DATA_PATH_CSV)
    else:
        raise FileNotFoundError(
            "model_df not found in memory and no placeholder file exists."
        )

model_df = model_df.copy()
model_df[DATE_COL] = pd.to_datetime(model_df[DATE_COL], errors="coerce")
model_df = model_df.dropna(subset=[DATE_COL])
model_df = model_df.sort_values([ENTITY_COL, DATE_COL]).reset_index(drop=True)

n_rows = len(model_df)
min_date = model_df[DATE_COL].min()
max_date = model_df[DATE_COL].max()
num_entities = model_df[ENTITY_COL].nunique()

print(f"Rows: {n_rows:,}")
print(f"Date range: {min_date.date()} to {max_date.date()}")
print(f"Unique entities: {num_entities}")

# Missingness report (top 20 columns)
missing = model_df.isna().mean().sort_values(ascending=False)
print("Missingness (top 20):")
print(missing.head(20))

# Daily granularity check
# Warn if gaps > 1 day exist per entity
print("Daily gap check (showing entities with gaps):")

gap_rows = []
for ent, g in model_df.groupby(ENTITY_COL):
    g = g.sort_values(DATE_COL)
    diffs = g[DATE_COL].diff().dt.days
    gap_count = (diffs > 1).sum()
    if gap_count > 0:
        gap_rows.append((ent, int(gap_count), float(diffs.max())))

if gap_rows:
    gap_df = pd.DataFrame(gap_rows, columns=["entity", "num_gaps", "max_gap_days"])
    print(gap_df.sort_values(["num_gaps", "max_gap_days"], ascending=False).head(10))
else:
    print("No gaps > 1 day detected.")


FileNotFoundError: model_df not found in memory and no placeholder file exists.

## 3) Feature engineering (leakage-safe)

Key points:
- All rolling features use `shift(1)` before `rolling`.
- Known-future exogenous features for horizon h are shifted by `-h`.
- No forward-filling of the target.


In [ ]:
# ----------------------------
# 3) Feature engineering (leakage-safe)
# ----------------------------
LAGS = [1, 2, 3, 7, 14, 28]
ROLL_WINDOWS = [7, 14, 28]
WEATHER_COLS = ["temp", "rain_1h", "wind_speed", "clouds_all", "humidity"]

FUTURE_KNOWN_COLS = []


def make_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create leakage-safe lag, rolling, and future-known features.
    Rolling stats are computed as shift(1).rolling(window).
    Future-known exogenous features are shifted by -h within each entity.
    """
    global FUTURE_KNOWN_COLS

    df = df.copy()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df.dropna(subset=[DATE_COL])
    df = df.sort_values([ENTITY_COL, DATE_COL]).reset_index(drop=True)

    # Basic transforms
    if "attendance" in df.columns:
        df["log_attendance"] = np.log1p(df["attendance"])

    # Ensure categorical types for stable encoding
    if ENTITY_COL in df.columns:
        df[ENTITY_COL] = df[ENTITY_COL].astype("category")
    if "season" in df.columns:
        df["season"] = df["season"].astype("category")
    if "dow" in df.columns:
        df["dow"] = df["dow"].astype("category")

    # Lag features for target and selected drivers
    driver_candidates = [
        "attendance",
        "log_attendance",
        "utilization",
        "availability",
        "scheduled_open_min",
        "nb_units_med",
    ]
    driver_cols = [c for c in driver_candidates if c in df.columns]

    for k in LAGS:
        df[f"y_lag_{k}"] = df.groupby(ENTITY_COL)[TARGET_COL].transform(lambda s: s.shift(k))
        for c in driver_cols:
            df[f"{c}_lag_{k}"] = df.groupby(ENTITY_COL)[c].transform(lambda s: s.shift(k))

    # Rolling window features (shift then roll)
    for w in ROLL_WINDOWS:
        df[f"y_roll_mean_{w}"] = df.groupby(ENTITY_COL)[TARGET_COL].transform(
            lambda s: s.shift(1).rolling(w).mean()
        )
        df[f"y_roll_std_{w}"] = df.groupby(ENTITY_COL)[TARGET_COL].transform(
            lambda s: s.shift(1).rolling(w).std()
        )
        for c in driver_cols:
            df[f"{c}_roll_mean_{w}"] = df.groupby(ENTITY_COL)[c].transform(
                lambda s: s.shift(1).rolling(w).mean()
            )

    # Identify future-known exogenous columns
    event_cols = [
        c for c in df.columns
        if ("event" in c.lower()) or ("holiday" in c.lower())
    ]

    future_known = []
    for c in ["dow", "season", "covid", "post_covid"]:
        if c in df.columns:
            future_known.append(c)
    for c in WEATHER_COLS:
        if c in df.columns:
            future_known.append(c)
    for c in event_cols:
        if c in df.columns:
            future_known.append(c)

    # Remove duplicates while preserving order
    FUTURE_KNOWN_COLS = list(dict.fromkeys(future_known))

    # Create horizon-specific future-known features
    for h in HORIZONS:
        for c in FUTURE_KNOWN_COLS:
            # Shift within entity to represent known exogenous values at t+h
            df[f"{c}_fut_h{h}"] = df.groupby(ENTITY_COL)[c].transform(lambda s: s.shift(-h))

    # Handle missing exogenous values without leaking target
    numeric_fill_cols = [
        c for c in driver_candidates + WEATHER_COLS
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
    ]
    for c in numeric_fill_cols:
        df[c] = df[c].fillna(df[c].median())

    # Event flags often behave like 0/1
    for c in event_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0)

    return df


features_df = make_features(model_df)
print("Feature engineering complete.")
print(f"Columns in features_df: {len(features_df.columns)}")
print("Future-known columns:", FUTURE_KNOWN_COLS)


## 4) Build training datasets per horizon

Each horizon dataset uses:
- Lags and rolling features computed at time t.
- Known-future features at time t+h (columns ending with `_fut_h{h}`).
- Target `y = wait_time_avg` shifted by -h within each entity.


In [ ]:
# ----------------------------
# 4) Build horizon-specific datasets
# ----------------------------

def build_horizon_dataset(df_features: pd.DataFrame, h: int):
    df = df_features.copy()
    df = df.sort_values([ENTITY_COL, DATE_COL]).reset_index(drop=True)

    y_col = f"y_t_plus_{h}"
    df[y_col] = df.groupby(ENTITY_COL)[TARGET_COL].transform(lambda s: s.shift(-h))

    # Select features without leakage
    exclude_cols = {TARGET_COL, y_col, DATE_COL}
    feature_cols = []
    for c in df.columns:
        if c in exclude_cols:
            continue
        # Only keep future-known features for the matching horizon
        if "_fut_h" in c and not c.endswith(f"_fut_h{h}"):
            continue
        feature_cols.append(c)

    # Build dataset and drop rows with missing caused by lags/rolls/leads
    data = df[feature_cols + [y_col, DATE_COL, ENTITY_COL]].dropna().reset_index(drop=True)

    X = data[feature_cols]
    y = data[y_col]
    meta = data[[DATE_COL, ENTITY_COL]].copy()
    meta["target_date"] = meta[DATE_COL] + pd.to_timedelta(h, unit="D")

    return X, y, meta, feature_cols


## 5) Encoding of ENTITY_DESCRIPTION_SHORT

LightGBM can handle categorical features natively.
If LightGBM is unavailable, we one-hot encode the entity column with aligned columns.


In [ ]:
# ----------------------------
# 5) Encoding utilities
# ----------------------------

def encode_features(X_train, X_val, X_test, use_lgbm: bool, entity_col: str):
    if use_lgbm:
        for df in [X_train, X_val, X_test]:
            if df is not None and entity_col in df.columns:
                df[entity_col] = df[entity_col].astype("category")
        return X_train, X_val, X_test, [entity_col]

    # XGBoost fallback: one-hot encode entity
    def _ohe(df):
        if df is None:
            return None
        return pd.get_dummies(df, columns=[entity_col], dummy_na=False)

    X_train_o = _ohe(X_train)
    X_val_o = _ohe(X_val)
    X_test_o = _ohe(X_test)

    if X_val_o is not None:
        X_val_o = X_val_o.reindex(columns=X_train_o.columns, fill_value=0)
    if X_test_o is not None:
        X_test_o = X_test_o.reindex(columns=X_train_o.columns, fill_value=0)

    return X_train_o, X_val_o, X_test_o, None


## 6) Time-based split + walk-forward validation

We create:
- A final holdout test block using the last 20 percent of origin dates.
- Walk-forward expanding-window splits for validation.


In [ ]:
# ----------------------------
# 6) Time-based split utilities
# ----------------------------

def make_date_splits(dates, test_size=0.2):
    dates = pd.to_datetime(pd.Series(dates)).sort_values().unique()
    n = len(dates)
    if n < 5:
        raise ValueError("Not enough unique dates for time split.")

    split_idx = int(n * (1 - test_size))
    split_idx = max(1, min(split_idx, n - 1))

    train_dates = dates[:split_idx]
    test_dates = dates[split_idx:]
    return train_dates, test_dates


def walk_forward_splits(dates, n_splits=3, min_train_frac=0.5):
    dates = pd.to_datetime(pd.Series(dates)).sort_values().unique()
    n = len(dates)
    min_train = max(1, int(n * min_train_frac))
    remaining = n - min_train
    if remaining <= 0:
        return []

    fold_size = max(1, remaining // n_splits)
    splits = []

    for i in range(n_splits):
        train_end = min_train + i * fold_size
        val_end = min(train_end + fold_size, n)
        if train_end >= val_end:
            break
        train_dates = dates[:train_end]
        val_dates = dates[train_end:val_end]
        splits.append((train_dates, val_dates))

    return splits


## 7) Model training (7 models, direct)

We train one model per horizon with early stopping on a time-based validation block.


In [ ]:
# ----------------------------
# 7) Training utilities
# ----------------------------

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def mae(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs(y_true - y_pred))


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.where(np.abs(y_true) < 1e-8, np.nan, np.abs(y_true))
    return np.nanmean(np.abs((y_true - y_pred) / denom))


def get_base_params(use_lgbm: bool):
    if use_lgbm:
        return dict(
            n_estimators=2000,
            learning_rate=0.05,
            num_leaves=64,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_samples=20,
            random_state=SEED,
            objective="regression",
        )
    return dict(
        n_estimators=2000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        objective="reg:squarederror",
        tree_method="hist",
    )


def train_point_model(X_train, y_train, X_val, y_val, use_lgbm, categorical_feature=None):
    params = get_base_params(use_lgbm)
    if use_lgbm:
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            categorical_feature=categorical_feature,
            callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)],
        )
        return model

    model = xgb.XGBRegressor(**params)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
        early_stopping_rounds=100,
    )
    return model


def train_quantile_model(X_train, y_train, X_val, y_val, q, use_lgbm, categorical_feature=None):
    if not use_lgbm:
        return None

    params = get_base_params(use_lgbm)
    params.update({"objective": "quantile", "alpha": q})
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="quantile",
        categorical_feature=categorical_feature,
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)],
    )
    return model


## 8) Train, validate, and evaluate across horizons

Includes walk-forward CV, final holdout evaluation, and quantile forecasts.


In [ ]:
# ----------------------------
# 8) Full training and evaluation
# ----------------------------
from joblib import dump

MODELS_DIR = Path("models")
OUTPUTS_DIR = Path("outputs")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

metrics_rows = []
cv_rows = []
test_pred_rows = []
feature_cols_by_h = {}
conformal_residuals_by_h = {}

for h in HORIZONS:
    print(f"
=== Horizon h={h} ===")
    X, y, meta, feature_cols = build_horizon_dataset(features_df, h)
    feature_cols_by_h[h] = feature_cols

    dates = meta[DATE_COL].sort_values().unique()
    train_dates, test_dates = make_date_splits(dates, test_size=0.2)

    train_mask = meta[DATE_COL].isin(train_dates)
    test_mask = meta[DATE_COL].isin(test_dates)

    X_train = X.loc[train_mask].reset_index(drop=True)
    y_train = y.loc[train_mask].reset_index(drop=True)
    X_test = X.loc[test_mask].reset_index(drop=True)
    y_test = y.loc[test_mask].reset_index(drop=True)
    meta_test = meta.loc[test_mask].reset_index(drop=True)

    # Time-based validation split for early stopping
    train_dates_sorted = pd.Series(train_dates).sort_values().unique()
    val_size = max(1, int(len(train_dates_sorted) * 0.1))
    if val_size >= len(train_dates_sorted):
        val_size = max(1, len(train_dates_sorted) // 2)

    val_dates = train_dates_sorted[-val_size:]
    tr_dates = train_dates_sorted[:-val_size]

    tr_mask = meta[DATE_COL].isin(tr_dates)
    val_mask = meta[DATE_COL].isin(val_dates)

    X_tr = X.loc[tr_mask].reset_index(drop=True)
    y_tr = y.loc[tr_mask].reset_index(drop=True)
    X_val = X.loc[val_mask].reset_index(drop=True)
    y_val = y.loc[val_mask].reset_index(drop=True)

    # Encode entity
    X_tr_enc, X_val_enc, X_test_enc, cat_feats = encode_features(
        X_tr.copy(), X_val.copy(), X_test.copy(), USE_LGBM, ENTITY_COL
    )

    # Train point model
    point_model = train_point_model(X_tr_enc, y_tr, X_val_enc, y_val, USE_LGBM, cat_feats)

    # Save point model and feature list
    dump(point_model, MODELS_DIR / f"model_h{h}.pkl")
    with open(MODELS_DIR / f"model_h{h}_features.json", "w", encoding="utf-8") as f:
        json.dump(feature_cols, f, indent=2)

    # Walk-forward CV on training dates
    cv_splits = walk_forward_splits(train_dates_sorted, n_splits=3, min_train_frac=0.5)
    for i, (cv_train_dates, cv_val_dates) in enumerate(cv_splits):
        cv_train_mask = meta[DATE_COL].isin(cv_train_dates)
        cv_val_mask = meta[DATE_COL].isin(cv_val_dates)

        X_cv_tr = X.loc[cv_train_mask].reset_index(drop=True)
        y_cv_tr = y.loc[cv_train_mask].reset_index(drop=True)
        X_cv_val = X.loc[cv_val_mask].reset_index(drop=True)
        y_cv_val = y.loc[cv_val_mask].reset_index(drop=True)

        X_cv_tr_enc, X_cv_val_enc, _, cv_cat_feats = encode_features(
            X_cv_tr.copy(), X_cv_val.copy(), None, USE_LGBM, ENTITY_COL
        )

        cv_model = train_point_model(X_cv_tr_enc, y_cv_tr, X_cv_val_enc, y_cv_val, USE_LGBM, cv_cat_feats)
        cv_pred = cv_model.predict(X_cv_val_enc)

        cv_rows.append({
            "horizon": h,
            "fold": i,
            "rmse": rmse(y_cv_val, cv_pred),
            "mae": mae(y_cv_val, cv_pred),
            "mape": mape(y_cv_val, cv_pred),
        })

    # Quantile or conformal models
    q10 = q50 = q90 = None
    if USE_LGBM:
        q_models = {}
        for q in [0.1, 0.5, 0.9]:
            q_model = train_quantile_model(X_tr_enc, y_tr, X_val_enc, y_val, q, USE_LGBM, cat_feats)
            q_models[q] = q_model
            dump(q_model, MODELS_DIR / f"model_h{h}_q{int(q*100):02d}.pkl")

        q10 = q_models[0.1].predict(X_test_enc)
        q50 = q_models[0.5].predict(X_test_enc)
        q90 = q_models[0.9].predict(X_test_enc)
    else:
        # Conformal intervals using validation residuals
        val_pred = point_model.predict(X_val_enc)
        residuals = y_val.values - val_pred
        q10_r, q50_r, q90_r = np.quantile(residuals, [0.1, 0.5, 0.9])
        conformal_residuals_by_h[h] = {
            "q10": float(q10_r),
            "q50": float(q50_r),
            "q90": float(q90_r),
        }

    # Test predictions
    yhat = point_model.predict(X_test_enc)
    if not USE_LGBM:
        # Apply conformal residual quantiles
        q10_r = conformal_residuals_by_h[h]["q10"]
        q50_r = conformal_residuals_by_h[h]["q50"]
        q90_r = conformal_residuals_by_h[h]["q90"]
        q10 = yhat + q10_r
        q50 = yhat + q50_r
        q90 = yhat + q90_r

    # Metrics
    horizon_rmse = rmse(y_test, yhat)
    horizon_mae = mae(y_test, yhat)
    horizon_mape = mape(y_test, yhat)
    coverage = np.mean((y_test.values >= q10) & (y_test.values <= q90))

    metrics_rows.append({
        "horizon": h,
        "rmse": horizon_rmse,
        "mae": horizon_mae,
        "mape": horizon_mape,
        "interval_coverage_10_90": coverage,
    })

    # Collect test predictions for output
    pred_df = meta_test.copy()
    pred_df["horizon"] = h
    pred_df["y_true"] = y_test.values
    pred_df["yhat"] = yhat
    pred_df["yhat_q10"] = q10
    pred_df["yhat_q50"] = q50
    pred_df["yhat_q90"] = q90
    test_pred_rows.append(pred_df)

# Save metrics and test predictions
metrics_df = pd.DataFrame(metrics_rows).sort_values("horizon")
metrics_df.to_csv(OUTPUTS_DIR / "metrics_by_horizon.csv", index=False)

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(OUTPUTS_DIR / "metrics_walk_forward_cv.csv", index=False)

all_test_preds = pd.concat(test_pred_rows, ignore_index=True)
all_test_preds.to_csv(OUTPUTS_DIR / "test_predictions.csv", index=False)

print("
Metrics by horizon:")
print(metrics_df)
print("
Walk-forward CV metrics (head):")
print(cv_df.head())


## 9) Evaluation details

Includes best and worst attractions by MAE and a basic calibration check.


In [ ]:
# ----------------------------
# 9) Evaluation details
# ----------------------------

def per_entity_metrics(df):
    rows = []
    for ent, g in df.groupby(ENTITY_COL):
        rows.append({
            "entity": ent,
            "mae": mae(g["y_true"], g["yhat"]),
            "rmse": rmse(g["y_true"], g["yhat"]),
        })
    return pd.DataFrame(rows).sort_values("mae")

print("
Per-horizon best and worst entities by MAE:")
for h in HORIZONS:
    g = all_test_preds[all_test_preds["horizon"] == h]
    ent_df = per_entity_metrics(g)
    print(f"
Horizon {h}")
    print("Best 10:")
    print(ent_df.head(10))
    print("Worst 10:")
    print(ent_df.tail(10))

# Calibration check for 10-90 intervals
coverage_by_h = (
    all_test_preds
    .groupby("horizon")
    .apply(lambda df: np.mean((df["y_true"] >= df["yhat_q10"]) & (df["y_true"] <= df["yhat_q90"])))
)
print("
Empirical coverage (10-90) by horizon:")
print(coverage_by_h)


## 10) Feature importance and diagnostics

Shows top 20 features by gain-based importance for each horizon model.


In [ ]:
# ----------------------------
# 10) Feature importance
# ----------------------------
import matplotlib.pyplot as plt
from joblib import load

for h in HORIZONS:
    model = load(MODELS_DIR / f"model_h{h}.pkl")
    with open(MODELS_DIR / f"model_h{h}_features.json", "r", encoding="utf-8") as f:
        feature_cols = json.load(f)

    importances = model.feature_importances_
    imp_df = pd.DataFrame({"feature": feature_cols, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False).head(20)

    print(f"
Top 20 features for horizon {h}:")
    print(imp_df)

    plt.figure(figsize=(8, 5))
    plt.barh(imp_df["feature"][::-1], imp_df["importance"][::-1])
    plt.title(f"Top 20 Feature Importances (h={h})")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()


## 11) Forecast the next 7 days (future inference)

Provide `future_exog_df` with one row per entity per future date containing
future-known columns such as season, dow, weather, and events.


In [ ]:
# ----------------------------
# 11) Forecast next 7 days
# ----------------------------

def make_future_frame(model_df, future_exog_df, horizons):
    """
    Build feature rows for forecasting the next horizons from the latest observed date.
    This function uses the latest historical date as forecast origin.
    """
    hist_features = make_features(model_df)
    last_date = hist_features[DATE_COL].max()

    base = hist_features[hist_features[DATE_COL] == last_date].copy()
    base = base.reset_index(drop=True)
    # Drop precomputed future-known columns to avoid merge suffixes
    fut_cols_existing = [c for c in base.columns if "_fut_h" in c]
    if fut_cols_existing:
        base = base.drop(columns=fut_cols_existing)
    base["forecast_origin_date"] = last_date

    future_exog_df = future_exog_df.copy()
    future_exog_df[DATE_COL] = pd.to_datetime(future_exog_df[DATE_COL], errors="coerce")

    # Ensure required future-known columns exist
    for c in FUTURE_KNOWN_COLS:
        if c not in future_exog_df.columns:
            future_exog_df[c] = np.nan

    all_rows = []
    for h in horizons:
        target_date = last_date + pd.to_timedelta(h, unit="D")
        fut = future_exog_df[future_exog_df[DATE_COL] == target_date].copy()

        rename_map = {c: f"{c}_fut_h{h}" for c in FUTURE_KNOWN_COLS if c in fut.columns}
        fut = fut.rename(columns=rename_map)

        # Keep only the columns needed for merge
        fut_cols = [ENTITY_COL] + list(rename_map.values())
        fut = fut[fut_cols]

        df_h = base.merge(fut, on=ENTITY_COL, how="left")
        df_h["target_date"] = target_date
        df_h["horizon"] = h
        all_rows.append(df_h)

    future_features = pd.concat(all_rows, ignore_index=True)
    return future_features

# Placeholder future exogenous dataframe
# future_exog_df must have columns: date, ENTITY_DESCRIPTION_SHORT, and future-known exogenous columns
# Example:
# future_exog_df = pd.read_csv("data/future_exog.csv")

if "future_exog_df" in globals():
    future_features = make_future_frame(model_df, future_exog_df, HORIZONS)

    forecast_rows = []
    for h in HORIZONS:
        model = load(MODELS_DIR / f"model_h{h}.pkl")
        with open(MODELS_DIR / f"model_h{h}_features.json", "r", encoding="utf-8") as f:
            feature_cols = json.load(f)

        df_h = future_features[future_features["horizon"] == h].copy()
        X_future = df_h[feature_cols]

        # Encode entity
        X_future_enc, _, _, _ = encode_features(X_future.copy(), None, None, USE_LGBM, ENTITY_COL)

        yhat = model.predict(X_future_enc)

        if USE_LGBM:
            q10 = load(MODELS_DIR / f"model_h{h}_q10.pkl").predict(X_future_enc)
            q50 = load(MODELS_DIR / f"model_h{h}_q50.pkl").predict(X_future_enc)
            q90 = load(MODELS_DIR / f"model_h{h}_q90.pkl").predict(X_future_enc)
        else:
            q10 = yhat + conformal_residuals_by_h[h]["q10"]
            q50 = yhat + conformal_residuals_by_h[h]["q50"]
            q90 = yhat + conformal_residuals_by_h[h]["q90"]

        out = pd.DataFrame({
            "entity": df_h[ENTITY_COL].values,
            "forecast_origin_date": df_h["forecast_origin_date"].values,
            "target_date": df_h["target_date"].values,
            "horizon": h,
            "yhat": yhat,
            "yhat_q10": q10,
            "yhat_q50": q50,
            "yhat_q90": q90,
        })
        forecast_rows.append(out)

    forecasts_next_7d = pd.concat(forecast_rows, ignore_index=True)
    forecasts_next_7d.to_csv(OUTPUTS_DIR / "forecasts_next_7d.csv", index=False)
    print("Saved next-7-days forecasts to outputs/forecasts_next_7d.csv")
else:
    print("future_exog_df not found. Skipping next-7-days forecast.")


## 12) Artifacts and rerun notes

This notebook saves:
- `models/model_h{h}.pkl` and `models/model_h{h}_features.json` for each horizon.
- Quantile models in `models/` if LightGBM is used.
- `outputs/metrics_by_horizon.csv`
- `outputs/metrics_walk_forward_cv.csv`
- `outputs/test_predictions.csv`
- `outputs/forecasts_next_7d.csv` when `future_exog_df` is provided.

Rerun notes:
- Ensure `model_df` is loaded or set a valid path at the top.
- Provide `future_exog_df` to generate next-7-days forecasts.
- All splits are time-ordered with no shuffling.
